In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

sold = pd.read_csv(
    PROCESSED_DIR / "crmls_sold_final_202401_202606.csv", low_memory=False
)
for field in ["CloseDate", "PurchaseContractDate", "ListingContractDate"]:
    sold[field] = pd.to_datetime(sold[field], errors="coerce")

print(sold.shape)


(447954, 65)


In [2]:
# Price ratios: negotiation strength vs final list price, and the full
# price-reduction history vs the original list price.
sold["sale_to_list_ratio"] = sold["ClosePrice"] / sold["ListPrice"]
sold["close_to_original_list_ratio"] = sold["ClosePrice"] / sold["OriginalListPrice"]

# Price per square foot (LivingArea=0 already set to missing in Week 4-5).
sold["price_per_sqft"] = sold["ClosePrice"] / sold["LivingArea"]

# Time-series keys from the close date.
sold["close_year"] = sold["CloseDate"].dt.year
sold["close_month"] = sold["CloseDate"].dt.month
sold["close_yrmo"] = sold["CloseDate"].dt.strftime("%Y-%m")

# Market speed metrics.
sold["listing_to_contract_days"] = (
    sold["PurchaseContractDate"] - sold["ListingContractDate"]
).dt.days
sold["contract_to_close_days"] = (
    sold["CloseDate"] - sold["PurchaseContractDate"]
).dt.days

new_metrics = [
    "sale_to_list_ratio", "close_to_original_list_ratio", "price_per_sqft",
    "listing_to_contract_days", "contract_to_close_days",
]
sold[["ListingKey", "ClosePrice"] + new_metrics].head(10)


,ListingKey,ClosePrice,sale_to_list_ratio,close_to_original_list_ratio,price_per_sqft,listing_to_contract_days,contract_to_close_days
0,551985747,240000.0,0.813559,0.480962,210.526316,777.0,65.0
1,522107581,815000.0,1.072510,1.072510,412.867275,114.0,919.0
2,510919001,810000.0,1.051948,1.094743,410.334347,255.0,778.0
3,1067652762,2100000.0,1.000000,1.000000,562.098501,0.0,48.0
4,1063453216,1950000.0,1.000000,1.000000,928.571429,32.0,34.0
5,1061988701,2340000.0,1.000000,NaN,958.230958,NaN,NaN
6,1061822266,1485000.0,0.958065,0.958065,927.545284,1.0,21.0
7,1061617422,1130000.0,1.131131,1.131131,529.026217,1.0,17.0
8,1060223286,1060000.0,1.009524,1.009524,552.947314,34.0,6.0
9,1060105211,2150000.0,1.000000,NaN,977.272727,NaN,NaN


In [3]:
# Sanity-check the new metrics before trusting them.
display(sold[new_metrics].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)


/Users/baixuezhang/Documents/IDX Exchange/real-estate-dashboard-project/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,count,mean,std,min,1%,25%,50%,75%,99%,max
sale_to_list_ratio,447952.0,1.080755,8.583178,9.591326e-07,0.842105,0.976410,1.000000,1.019512,1.257143,1.153846e+03
close_to_original_list_ratio,447133.0,inf,NaN,9.207366e-07,0.747374,0.953721,0.995653,1.019355,1.284895,inf
price_per_sqft,447536.0,646.246830,5047.470524,4.981497e-04,160.743950,368.019819,537.303217,732.536496,1895.049565,1.164067e+06
listing_to_contract_days,447229.0,45.297914,66.826857,0.000000e+00,0.000000,10.000000,25.000000,58.000000,277.000000,1.465700e+04
contract_to_close_days,447517.0,31.548343,25.623293,0.000000e+00,0.000000,21.000000,29.000000,36.000000,117.000000,1.970000e+03


In [4]:
# How many list prices are invalid (would poison the ratios)?
for col in ["ListPrice", "OriginalListPrice"]:
    s = sold[col]
    print(f"{col}: zero={int(s.eq(0).sum())}, negative={int(s.lt(0).sum())}, "
          f"missing={int(s.isna().sum())}, under $1,000={int(s.between(0.01, 1000).sum())}")

# Look at the worst ratio offenders to see what bad list prices look like.
display(
    sold.loc[
        sold["sale_to_list_ratio"].gt(5) | sold["sale_to_list_ratio"].lt(0.1),
        ["ListingKey", "ClosePrice", "ListPrice", "OriginalListPrice",
         "sale_to_list_ratio", "City"],
    ].head(15)
)


ListPrice: zero=0, negative=0, missing=0, under $1,000=2
OriginalListPrice: zero=2, negative=0, missing=819, under $1,000=150


,ListingKey,ClosePrice,ListPrice,OriginalListPrice,sale_to_list_ratio,City
7644,1046397327,3.150000e+04,339500.0,349500.0,0.092784,Gustine
24280,1065201431,6.950000e+05,695.0,695.0,1000.000000,Anaheim
36705,1052432731,4.450000e+04,449900.0,549900.0,0.098911,San Bernardino
44885,1063269497,8.150000e+06,774900.0,774900.0,10.517486,Bellflower
58610,1073171652,6.132024e+06,488000.0,488000.0,12.565623,Pomona
62469,1068116132,6.480000e+06,638000.0,638000.0,10.156740,San Bernardino
63323,1067634245,5.598999e+06,598999.0,598999.0,9.347259,Lake Elsinore
65947,1065242616,5.077777e+04,545000.0,545000.0,0.093170,Commerce
72822,1061309278,9.550000e+06,929000.0,929888.0,10.279871,Chino Hills
72974,1061235712,1.336500e+05,1350000.0,1350000.0,0.099000,Temecula


In [5]:
# Two verified price errors found via the ratio check:
# - Brentwood row (ListingKey 1059929060): ClosePrice $345 captured only the
#   trailing digits of its $784,345 list price. The true close price cannot
#   be reconstructed, and close price is essential -> delete the row.
# - Anaheim row (ListingKey 1065201431): list price lost its thousands
#   (695 vs a $695,000 close, an at-list sale) -> restore the three zeros.
rows_before = len(sold)
sold = sold.loc[~sold["ListingKey"].eq(1059929060)].copy()
print(f"Deleted Brentwood row: {rows_before:,} -> {len(sold):,}")

anaheim = sold["ListingKey"].eq(1065201431)
sold.loc[anaheim, ["ListPrice", "OriginalListPrice"]] = 695000.0
print("Anaheim list price repaired:",
      sold.loc[anaheim, ["ClosePrice", "ListPrice", "OriginalListPrice"]].to_dict("records"))

# Recompute the ratios so the repaired row gets real values.
sold["sale_to_list_ratio"] = sold["ClosePrice"] / sold["ListPrice"]
sold["close_to_original_list_ratio"] = sold["ClosePrice"] / sold["OriginalListPrice"]

Deleted Brentwood row: 447,954 -> 447,953
Anaheim list price repaired: [{'ClosePrice': 695000.0, 'ListPrice': 695000.0, 'OriginalListPrice': 695000.0}]


**Ratio guard:** prices below \$1,000 are placeholder or unit errors
(e.g., an original list price of \$500), so ratios built on them are set to
missing while the rows are kept. Extreme but legitimately-computed ratios
(digit typos such as a 10x close price) are deferred to Week 7 IQR filtering.

In [6]:
# Invalidate ratios built on placeholder-grade prices. A California
# residential price below $1,000 is a data-entry or unit error, not a real
# price — so any ratio using it is meaningless. Rows are kept; only the
# ratio is set to missing.
MIN_VALID_PRICE = 1000

ratio_specs = [
    ("ClosePrice", "ListPrice", "sale_to_list_ratio"),
    ("ClosePrice", "OriginalListPrice", "close_to_original_list_ratio"),
]
for numerator, denominator, ratio in ratio_specs:
    invalid = (
        sold[numerator].lt(MIN_VALID_PRICE).fillna(True)
        | sold[denominator].lt(MIN_VALID_PRICE).fillna(True)
    )
    sold.loc[invalid, ratio] = pd.NA
    print(f"{ratio}: {int(invalid.sum())} rows set to missing")

# Re-check the distributions: inf gone, medians unchanged. sale_to_list_ratio = ClosePrice / ListPrice, close_to_original_list_ratio = ClosePrice / OriginalListPrice.
display(
    sold[["sale_to_list_ratio", "close_to_original_list_ratio"]]
    .describe(percentiles=[0.01, 0.5, 0.99])
    .T
)

sale_to_list_ratio: 8 rows set to missing
close_to_original_list_ratio: 157 rows set to missing


,count,mean,std,min,1%,50%,99%,max
sale_to_list_ratio,447943.0,1.076314,8.319686,0.004150,0.842105,1.000000,1.257143,1153.846154
close_to_original_list_ratio,446975.0,1.129729,10.897452,0.000513,0.747493,0.995624,1.280000,1153.846154


In [7]:
# connect with school district data

import geopandas as gpd

# --- School district spatial join: trial run on a 50k sample first. ---
SCHOOL_DISTRICT_SHP = PROJECT_ROOT / "data" / "raw" / "school_districts" / "DistrictAreas2425.shp"

# Load district polygons; keep only what we need. The shapefile ships in
# Web Mercator (EPSG:3857) while our coordinates are lat/lon (EPSG:4326),
# so reproject before joining.
districts = gpd.read_file(SCHOOL_DISTRICT_SHP)[["DistrictNa", "DistrictTy", "geometry"]]
districts = districts.to_crs(4326)
print(f"Districts: {len(districts)}, types: {districts['DistrictTy'].value_counts().to_dict()}")

# Sample 50k rows with coordinates for the trial.
sample = sold.loc[
    sold["Latitude"].notna() & sold["Longitude"].notna(),
    ["ListingKey", "Latitude", "Longitude", "City", "HighSchoolDistrict"],
].head(50000)

points = gpd.GeoDataFrame(
    sample,
    geometry=gpd.points_from_xy(sample["Longitude"], sample["Latitude"]),
    crs=4326,
)

# Point-in-polygon join: each point matches the district polygon containing it.
matched = gpd.sjoin(points, districts, how="left", predicate="within")

# A point falls in either ONE Unified district or an Elementary + High pair,
# so pivot district type into columns to keep one row per property.
district_cols = (
    matched.pivot_table(
        index="ListingKey", columns="DistrictTy", values="DistrictNa", aggfunc="first"
    )
    .rename(columns={
        "Unified": "school_district_unified",
        "Elementary": "school_district_elementary",
        "High": "school_district_high",
    })
)

trial = sample.merge(district_cols, on="ListingKey", how="left")
match_rate = trial[district_cols.columns].notna().any(axis=1).mean()
print(f"Match rate: {match_rate * 100:.2f}%")

# Eyeball check: does the assigned district look right for the city, and
# does it broadly agree with the agent-entered HighSchoolDistrict field?
display(trial.head(15))



Districts: 937, types: {'Elementary': 516, 'Unified': 345, 'High': 76}
Match rate: 99.99%


,ListingKey,Latitude,Longitude,City,HighSchoolDistrict,school_district_elementary,school_district_high,school_district_unified
0,551985747,37.566106,-122.327954,San Mateo,Other,San Mateo-Foster City,San Mateo Union High,NaN
1,522107581,32.659315,-117.096922,National City,NaN,National Elementary,Sweetwater Union High,NaN
2,510919001,32.659284,-117.097174,National City,NaN,National Elementary,Sweetwater Union High,NaN
3,1067652762,35.277199,-120.646786,San Luis Obispo,San Luis Coastal Unified,NaN,NaN,San Luis Coastal Unified
4,1063453216,39.419080,-123.814842,Fort Bragg,NaN,NaN,NaN,Fort Bragg Unified
5,1061988701,37.325454,-121.780303,San Jose,Other,Evergreen Elementary,East Side Union High,NaN
6,1061822266,32.715089,-117.170568,San Diego,San Diego Unified,NaN,NaN,San Diego Unified
7,1061617422,34.070215,-118.258618,Los Angeles,NaN,NaN,NaN,Los Angeles Unified
8,1060223286,34.107850,-118.219273,Los Angeles,NaN,NaN,NaN,Los Angeles Unified
9,1060105211,37.589762,-122.399887,Millbrae,Other,Millbrae Elementary,San Mateo Union High,NaN


In [8]:
# --- Full-dataset school district join, merged back into `sold`. ---

# Join on unique coordinates (many condos share a point), then map back —
# much faster than joining all 447k rows directly.
coords = (
    sold.loc[
        sold["Latitude"].notna() & sold["Longitude"].notna(),
        ["Latitude", "Longitude"],
    ]
    .drop_duplicates()
)
points = gpd.GeoDataFrame(
    coords,
    geometry=gpd.points_from_xy(coords["Longitude"], coords["Latitude"]),
    crs=4326,
)
matched = gpd.sjoin(points, districts, how="left", predicate="within")

district_lookup = (
    matched.pivot_table(
        index=["Latitude", "Longitude"],
        columns="DistrictTy",
        values="DistrictNa",
        aggfunc="first",
    )
    .rename(columns={
        "Unified": "school_district_unified",
        "Elementary": "school_district_elementary",
        "High": "school_district_high",
    })
    .reset_index()
)

# Drop the columns first if this cell is re-run, so merge stays clean.
sold = sold.drop(
    columns=[c for c in sold.columns if c.startswith("school_district_")],
    errors="ignore",
)
sold = sold.merge(district_lookup, on=["Latitude", "Longitude"], how="left")

print(f"Unique coordinates joined: {len(coords):,}")
print(f"sold is now {sold.shape[0]:,} rows x {sold.shape[1]} columns")

Unique coordinates joined: 408,089
sold is now 447,953 rows x 75 columns


In [9]:
district_cols = [
    "school_district_unified",
    "school_district_elementary",
    "school_district_high",
]

# Per-column missing share.
for col in district_cols:
    pct = sold[col].isna().mean() * 100
    print(f"{col}: {pct:.2f}% missing")

# Structural view: how the three columns combine per row.
no_district = sold[district_cols].isna().all(axis=1)
has_unified = sold["school_district_unified"].notna()
has_pair = (
    sold["school_district_elementary"].notna()
    & sold["school_district_high"].notna()
)
print()
print(f"No district at all : {no_district.mean() * 100:.2f}%  ({int(no_district.sum()):,} rows)")
print(f"Unified district   : {has_unified.mean() * 100:.2f}%")
print(f"Elementary + High  : {has_pair.mean() * 100:.2f}%")
print(f"Missing coordinates: {sold['Latitude'].isna().mean() * 100:.2f}%  (for comparison)")

school_district_unified: 25.13% missing
school_district_elementary: 75.48% missing
school_district_high: 75.98% missing

No district at all : 1.02%  (4,548 rows)
Unified district   : 74.87%
Elementary + High  : 24.02%
Missing coordinates: 0.99%  (for comparison)


In [10]:
# Do Elementary and High districts pair up as expected?
print("有elementary但没有high的行数:", int((sold["school_district_elementary"].notna() & sold["school_district_high"].isna()).sum()))
print("有high但没有elementary的行数:", int((sold["school_district_high"].notna() & sold["school_district_elementary"].isna()).sum()))

# The "elementary without high" rows should carry a Unified district for
# their high-school level (Pattern C, mixed districts).
elem_no_high = sold["school_district_elementary"].notna() & sold["school_district_high"].isna()
print("这些行里同时有unified的:", int((elem_no_high & sold["school_district_unified"].notna()).sum()))
print("这些行里没有unified的  :", int((elem_no_high & sold["school_district_unified"].isna()).sum()))

有elementary但没有high的行数: 2214
有high但没有elementary的行数: 0
这些行里同时有unified的: 1811
这些行里没有unified的  : 403


### School District Types: Three Valid Combinations

California public schools are organized into three district types by grade
level: **Unified** (K-12), **Elementary** (through grade 8), and **High**
(grades 9-12). A property's three district columns therefore fall into one
of three patterns, not two:

| Pattern | unified | elementary | high | Meaning | Share |
| --- | --- | --- | --- | --- | --- |
| A. Unified | filled | empty | empty | One district covers K-12 | ~74.9% |
| B. Elementary + High | empty | filled | filled | Separate districts hand off at grade 9 | ~23.5% |
| C. Mixed | filled | filled | empty | Elementary district for K-8; high school folded into the Unified district | ~0.5% (2,214 rows) |

Verified against the data: 0 rows have "high without elementary", and the
2,214 "elementary without high" rows all carry a Unified district for their
high-school level (Pattern C). So every geocoded property has complete
school-district coverage — the ~75% "missing" rate on the elementary and
high columns is structural (not-applicable), not true missing data. The only
genuine gap is the ~1% of rows lacking coordinates.

In [11]:
def summarize_by(df, group_col, min_count=10):
    """Summary stats for the key market metrics, grouped by one dimension."""
    summary = (
        df.groupby(group_col)
        .agg(
            homes_sold=("ClosePrice", "size"),
            median_close_price=("ClosePrice", "median"),
            median_price_per_sqft=("price_per_sqft", "median"),
            median_dom=("DaysOnMarket", "median"),
            median_sale_to_list=("sale_to_list_ratio", "median"),
            median_listing_to_contract=("listing_to_contract_days", "median"),
        )
        .round(2)
    )
    # Drop tiny segments whose medians are statistically noisy.
    summary = summary[summary["homes_sold"] >= min_count]
    return summary.sort_values("homes_sold", ascending=False)


# 1. By property subtype.
by_subtype = summarize_by(sold, "PropertySubType")
print("=== By PropertySubType ===")
display(by_subtype)

# 2. By county.
by_county = summarize_by(sold, "CountyOrParish")
print("=== By County (top 15) ===")
display(by_county.head(15))

# 3. Listing office performance (competitive intelligence).
by_list_office = summarize_by(sold, "ListOfficeName", min_count=50)
print("=== Top 15 Listing Offices by Volume ===")
display(by_list_office.head(15))

=== By PropertySubType ===


,homes_sold,median_close_price,median_price_per_sqft,median_dom,median_sale_to_list,median_listing_to_contract
PropertySubType,,,,,,
SingleFamilyResidence,335582,896000.0,533.22,17.0,1.00,23.0
Condominium,73579,626888.0,564.10,24.0,1.00,30.0
Townhouse,26236,800000.0,560.72,17.0,1.00,22.0
ManufacturedOnLand,5795,320000.0,225.00,30.0,0.99,39.0
Duplex,2492,910000.0,542.91,21.0,1.00,31.0
StockCooperative,1765,360000.0,396.21,20.5,1.00,31.0
Cabin,508,240500.0,287.76,46.5,0.96,56.0
Triplex,379,1135000.0,473.95,29.0,0.98,43.0
MixedUse,222,687500.0,433.95,52.5,0.96,67.0


=== By County (top 15) ===


,homes_sold,median_close_price,median_price_per_sqft,median_dom,median_sale_to_list,median_listing_to_contract
CountyOrParish,,,,,,
Los Angeles,111215,905000.0,608.95,19.0,1.00,25.0
Riverside,62119,600000.0,321.10,30.0,1.00,38.0
San Diego,55541,900000.0,590.77,15.0,1.00,21.0
Orange,50389,1180000.0,674.10,14.0,1.00,25.0
San Bernardino,41585,532000.0,331.99,24.0,1.00,32.0
Alameda,21202,1140000.0,700.87,14.0,1.02,14.0
Contra Costa,20466,825000.0,521.87,14.0,1.00,15.0
Santa Clara,19650,1600000.0,965.80,10.0,1.02,10.0
Ventura,14020,870000.0,516.82,28.0,1.00,35.0


=== Top 15 Listing Offices by Volume ===


,homes_sold,median_close_price,median_price_per_sqft,median_dom,median_sale_to_list,median_listing_to_contract
ListOfficeName,,,,,,
Compass,31721,1350000.0,757.24,14.0,1.00,17.0
Coldwell Banker Realty,20207,1200000.0,694.05,16.0,1.00,21.0
Keller Williams Realty,8762,875000.0,540.45,15.0,1.00,21.0
First Team Real Estate,6252,960000.0,609.32,13.0,1.00,26.0
Berkshire Hathaway HomeServices California Properties,5818,950000.0,598.53,23.0,0.99,29.0
eXp Realty of California Inc,5227,790000.0,531.91,19.0,1.00,26.0
Real Broker,5177,849900.0,558.26,15.0,1.00,23.0
Intero Real Estate Services,4809,1376000.0,847.91,11.0,1.02,11.0
Equity Union,3929,852000.0,475.71,28.0,0.99,33.0


In [12]:
# --- Save the feature dataset and the segment summary reports. ---
REPORT_DIR = PROJECT_ROOT / "data" / "reports" / "week6_feature_engineering"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_FILE = PROCESSED_DIR / "crmls_sold_features_202401_202606.csv"
sold.to_csv(FEATURES_FILE, index=False)
print(f"Saved feature dataset: {FEATURES_FILE}")
print(f"  {sold.shape[0]:,} rows x {sold.shape[1]} columns")

# Segment summaries (index=True keeps the group label as the first column).
by_subtype.to_csv(REPORT_DIR / "summary_by_property_subtype.csv")
by_county.to_csv(REPORT_DIR / "summary_by_county.csv")
by_list_office.head(50).to_csv(REPORT_DIR / "summary_by_listing_office.csv")
print(f"Saved 3 segment summaries to: {REPORT_DIR}")

Saved feature dataset: /Users/baixuezhang/Documents/IDX Exchange/real-estate-dashboard-project/data/processed/crmls_sold_features_202401_202606.csv
  447,953 rows x 75 columns
Saved 3 segment summaries to: /Users/baixuezhang/Documents/IDX Exchange/real-estate-dashboard-project/data/reports/week6_feature_engineering
